In [1]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import librosa
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report


In [2]:
def extract_features(file):
    audio, sr = librosa.load(file, sr=None)
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13)
    return np.mean(mfcc.T, axis=0)


In [3]:
# Reproducibility
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cpu')

In [5]:
# Dataset paths (Kaggle)
data_dir_human = r"D:\human-nonhuman\human"
data_dir_nonhuman = r"D:\human-nonhuman\nonhuman"

# Audio
SR = 16000
DURATION = 22.0
MAX_LEN = int(SR * DURATION)

# MFCC
N_MFCC = 40
N_FFT = 1024
HOP_LENGTH = 256
INCLUDE_DELTAS = True

# Training
BATCH_SIZE = 16
EPOCHS = 12
LR = 1e-3
WEIGHT_DECAY = 1e-4

# Early stopping
PATIENCE = 3


In [6]:
def load_file(path):
    y, _ = librosa.load(path, sr=SR, mono=True)

    if len(y) < MAX_LEN:
        y = np.pad(y, (0, MAX_LEN - len(y)), mode="constant")
    else:
        y = y[:MAX_LEN]

    mfcc = librosa.feature.mfcc(
        y=y,
        sr=SR,
        n_mfcc=N_MFCC,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH
    )

    if INCLUDE_DELTAS:
        delta = librosa.feature.delta(mfcc)
        delta2 = librosa.feature.delta(mfcc, order=2)
        feat = np.concatenate([mfcc, delta, delta2], axis=0)
    else:
        feat = mfcc

    return feat.astype(np.float32)


In [7]:
def collect_paths():
    items = []
    for f in os.listdir(data_dir_human):
        if f.lower().endswith(".mp3"):
            items.append((os.path.join(data_dir_human, f), 1))
    for f in os.listdir(data_dir_nonhuman):
        if f.lower().endswith(".mp3"):
            items.append((os.path.join(data_dir_nonhuman, f), 0))
    return items


In [8]:
class MFCCDataset(Dataset):
    def __init__(self, items):
        self.items = items

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        path, label = self.items[idx]
        feat = load_file(path)
        x = torch.tensor(feat).unsqueeze(0)  # (1, C, T)
        y = torch.tensor(label, dtype=torch.long)
        return x, y


In [9]:
class SmallCNN(nn.Module):
    def __init__(self, n_feats):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d((2, 2)),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d((2, 2)),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.fc = nn.Linear(64, 2)

    def forward(self, x):
        x = self.net(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)


In [10]:
items = collect_paths()
paths, labels = zip(*items)

train_items, test_items = train_test_split(
    items, test_size=0.3, random_state=seed, stratify=labels
)

train_ds = MFCCDataset(train_items)
test_ds = MFCCDataset(test_items)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)


In [11]:
dummy_feat = load_file(train_items[0][0])
N_FEATS = dummy_feat.shape[0]

model = SmallCNN(N_FEATS).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)


In [12]:
best_val = float("inf")
patience_ctr = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * xb.size(0)

    train_loss /= len(train_loader.dataset)

    model.eval()
    preds, trues = [], []
    val_loss = 0.0

    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            val_loss += loss.item() * xb.size(0)
            preds.append(torch.argmax(logits, 1).cpu().numpy())
            trues.append(yb.cpu().numpy())

    val_loss /= len(test_loader.dataset)
    preds = np.concatenate(preds)
    trues = np.concatenate(trues)

    acc = accuracy_score(trues, preds)
    f1 = f1_score(trues, preds)

    print(f"Epoch {epoch:02d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | acc={acc:.4f} | f1={f1:.4f}")

    if val_loss < best_val:
        best_val = val_loss
        patience_ctr = 0
        torch.save(model.state_dict(), "best_model.pt")
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print("Early stopping")
            break


Epoch 01 | train_loss=0.5280 | val_loss=0.4616 | acc=0.7870 | f1=0.8217


KeyboardInterrupt: 

In [13]:
model.load_state_dict(torch.load("best_model.pt"))
model.eval()

preds, trues = [], []

with torch.no_grad():
    for xb, yb in test_loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        preds.append(torch.argmax(logits, 1).cpu().numpy())
        trues.append(yb.cpu().numpy())

preds = np.concatenate(preds)
trues = np.concatenate(trues)

print("Accuracy:", accuracy_score(trues, preds))
print("F1-score:", f1_score(trues, preds))
print(confusion_matrix(trues, preds))
print(classification_report(trues, preds, target_names=["nonhuman", "human"]))


Accuracy: 0.7870216306156406
F1-score: 0.8217270194986073
[[178 122]
 [  6 295]]
              precision    recall  f1-score   support

    nonhuman       0.97      0.59      0.74       300
       human       0.71      0.98      0.82       301

    accuracy                           0.79       601
   macro avg       0.84      0.79      0.78       601
weighted avg       0.84      0.79      0.78       601

